# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's inspect the available record sets, their `@id`s, and the fields (columns) within each record set.

Note: All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets and their fields
print("Record Sets Available:")
record_sets = list(dataset.record_sets())
all_field_ids = {}
for record_set in record_sets:
    print(f"- RecordSet name: {record_set.name if hasattr(record_set, 'name') else ''}")
    print(f"  @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields (by @id):")
        for field in record_set.fields:
            print(f"    - {field.name if hasattr(field, 'name') else ''} (@id: {field.id})")
        # Collect all field ids per record_set
        all_field_ids[record_set.id] = [field.id for field in record_set.fields]
    print()

In [ ]:
# For demonstration, preview the records for the first record set
if record_sets:
    selected_record_set_id = record_sets[0].id
    print(f"Showing the first 2 records for record set '{selected_record_set_id}':")
    for i, x in enumerate(dataset.records(record_set=selected_record_set_id)):
        pprint(x)
        if i >= 1:
            break

## 3. Data Extraction
Load records from each record set into a DataFrame. Use the `@id`s collected in the previous step.

In [ ]:
# Extract data from all available record sets as DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set '{record_set_id}' loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# List columns for the first record set and preview its head
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nFields for record set '{main_rs_id}':\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
#--- EDA Example ---#
# For demonstration, pick the main record set and find a numeric field to analyze.
main_df = dataframes[main_rs_id]

# List autofilled numeric columns if present
numeric_candidate = None
for col in main_df.columns:
    # Try to convert column to numeric and see if not null
    col_data = pd.to_numeric(main_df[col], errors='coerce')
    if col_data.notnull().sum() > 0 and col_data.notnull().sum() == len(col_data):
        numeric_candidate = col
        break

if numeric_candidate:
    print(f"Numeric field for analysis: '{numeric_candidate}'")
    main_df[numeric_candidate] = pd.to_numeric(main_df[numeric_candidate], errors='coerce')
    threshold = main_df[numeric_candidate].mean()
    print(f"Filtering records with {numeric_candidate} > {threshold:.2f}")
    filtered_df = main_df[main_df[numeric_candidate] > threshold].copy()
    print(f"Filtered records:")
    display(filtered_df.head())
    normcol = f"{numeric_candidate}_normalized"
    filtered_df[normcol] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
    print(f"\nNormalized values:")
    display(filtered_df[[numeric_candidate, normcol]].head())

    # Try to group by a categorical field
    group_field = None
    for col in main_df.columns:
        if col != numeric_candidate and main_df[col].nunique() < len(main_df) / 2:
            group_field = col
            break
    if group_field:
        print(f"\nGrouping by field: '{group_field}'")
        grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No suitable grouping field found.")
else:
    print("No fully numeric fields found in the main record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For tabular data, we can use histograms and boxplots for numeric fields, or bar plots for categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidate:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_candidate].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_candidate}'")
    plt.xlabel(numeric_candidate)
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_candidate])
        plt.title(f"{numeric_candidate} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and explored clinical and pathological data on cancer survivors with second primary colorectal cancer using the `mlcroissant` library.
- Inspected record set structure via `@id` and loaded the data into pandas DataFrames.
- Conducted basic EDA including value filtering, normalization, and grouping.
- Visualized the distributions of numeric fields and explored relationships between key variables.

For more advanced analyses (e.g., statistical modeling, advanced visualizations), refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) and the [original dataset documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).